In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
DATA = Path("data")

## Images of a bar
A bright bar on a dark background, 41 x 41 pixels, through a chosen point at a chosen angle.

In [ ]:
N = 41
yy, xx = np.mgrid[0:N, 0:N]

def bar(angle_deg, shift=0.0, width=1.5, length=14):
    """Bar centred at (20, 20 + shift); angle 90 = vertical, 0 = horizontal."""
    a = np.deg2rad(angle_deg)
    dx, dy = xx - (20 + shift), yy - 20
    along = dx * np.cos(a) - dy * np.sin(a)          # distance along the bar
    across = dx * np.sin(a) + dy * np.cos(a)         # distance across the bar
    return ((np.abs(across) <= width) & (np.abs(along) <= length)).astype(float)

## The model simple cell: a 7 x 7 vertical-bar filter
Positive in the middle column, negative on both sides: it answers a bright vertical line in its small window.

In [ ]:
K = np.zeros((7, 7))
K[:, 2:5] = 1.0          # centre (3 columns)
K[:, :2] = -0.75         # left flank
K[:, 5:] = -0.75         # right flank
print(K)

def simple_cell(img, r=20, c=20):
    """Response of the filter placed at one position (its receptive field), then ReLU."""
    patch = img[r - 3:r + 4, c - 3:c + 4]
    return max(0.0, float((patch * K).sum()))

def complex_cell(img, r=20, c=20, reach=8):
    """Maximum of the simple-cell responses over a neighbourhood: a larger receptive field."""
    return max(simple_cell(img, rr, cc) for rr in range(r - reach, r + reach + 1) for cc in range(c - reach, c + reach + 1))

## Orientation tuning: rotate the bar from horizontal (0) through vertical (90) back to horizontal (180)

In [ ]:
angles = np.arange(0, 181, 5)
tuning = pd.DataFrame(dict(angle=angles,
                           simple=[simple_cell(bar(a)) for a in angles],
                           complex=[complex_cell(bar(a)) for a in angles]))
tuning["simple"] /= tuning.simple.max()
tuning["complex"] /= tuning.complex.max()
print(tuning.iloc[::3].round(2).to_string(index=False))
tuning.to_csv(DATA / "tuning.csv", index=False)

## Position: shift a vertical bar sideways

In [ ]:
shifts = np.arange(-8, 9)
pos = pd.DataFrame(dict(shift=shifts,
                        simple=[simple_cell(bar(90, s)) for s in shifts],
                        complex=[complex_cell(bar(90, s)) for s in shifts]))
top = pos[["simple", "complex"]].to_numpy().max()
pos[["simple", "complex"]] /= top
print(pos.round(2).to_string(index=False))
pos.to_csv(DATA / "position.csv", index=False)
np.save(DATA / "bars.npy", np.stack([bar(0), bar(45), bar(90), bar(90, 5)]))